# Ensemble Model Testing

This notebook tests the K-fold ensemble loading functionality.

## Features Tested:
- Loading ensemble model from K-fold checkpoints
- Weighted voting based on fold performance (F1 scores)
- Automatic checkpoint search across multiple locations
- Performance validation (95% threshold)
- Automatic retraining for missing/degraded checkpoints

In [1]:
from GradientGang.Pipeline.FinalPipeline import FinalPipeline
import torch
import os
%load_ext autoreload
%autoreload 2

## Initialize FinalPipeline

Set up the pipeline with your project configuration.

In [ ]:
# Initialize FinalPipeline with parameters
import os
import dotenv

dotenv.load_dotenv()
database_url = os.getenv("DATABASE_URL")

data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
    'use_kfold': True,
    'n_folds': 5,  # Should match your training configuration
}

params = {
    "database_url": database_url,
    "data_params": data_params,
    'project_name': 'pirate_pain_classification',
    'study_name': '_theBeastComputerTPE_AugmentedReduced_macro',
    'submission_path': '../Submissions/',
}

pipeline = FinalPipeline(params)

Storage: postgresql://postgres...
✓ Database configuration loaded
✓ Database initialized successfully!


[I 2025-11-15 20:01:30,116] Using an existing study with name 'pirate_pain_classification_theBeastComputerTPE_AugmentedReduced_macro' instead of creating a new one.


✓ Study created/loaded successfully!
Study name: pirate_pain_classification_theBeastComputerTPE_AugmentedReduced_macro
Sampler: TPESampler
Pruner: PatientPruner
Storage: Database
Total trials: 79
✓ Study initialized successfully!
✓ FinalPipeline initialized successfully!


## Check Study Information

Verify that your Optuna study has completed trials with K-fold results.

In [3]:
# Display study summary
pipeline.study_summary()

# Get best trial
best_trial = pipeline.study.best_trial
print(f"\nBest trial number: {best_trial.number}")
print(f"Best F1 score: {best_trial.value:.4f}")

# Check if fold_scores exist
if 'fold_scores' in best_trial.user_attrs:
    fold_scores_str = best_trial.user_attrs['fold_scores']
    fold_scores = [float(x) for x in fold_scores_str.split(',')]
    print(f"\nFold scores: {fold_scores}")
    print(f"Number of folds: {len(fold_scores)}")
else:
    print("\n⚠️ WARNING: Best trial does not have 'fold_scores' attribute!")
    print("Make sure you ran training with K-fold cross-validation.")

Study name: pirate_pain_classification_theBeastComputerTPE_AugmentedReduced_macro
Direction: 2
Total trials: 79
Completed trials: 61
Failed trials: 5
Pruned trials: 6
Running trials: 7

✓ Best trial: 21
✓ Best F1 score: 0.9019

Top 5 trials:
  1. Trial 21: F1=0.9019 | Autoencoder | Recurrent
  2. Trial 39: F1=0.8981 | Autoencoder | Recurrent
  3. Trial 27: F1=0.8947 | Autoencoder | Recurrent
  4. Trial 64: F1=0.8943 | Autoencoder | MultiScaleCNN
  5. Trial 22: F1=0.8914 | Autoencoder | Recurrent

📊 Trial states (last 10):
  ✓ Trial 69: COMPLETE | F1=0.8731
  ⟳ Trial 70: RUNNING | N/A
  ⟳ Trial 71: RUNNING | N/A
  ⟳ Trial 72: RUNNING | N/A
  ⟳ Trial 73: RUNNING | N/A
  ⟳ Trial 74: RUNNING | N/A
  ⟳ Trial 75: RUNNING | N/A
  ✓ Trial 76: COMPLETE | F1=0.8804
  ⟳ Trial 77: RUNNING | N/A
  ✗ Trial 78: FAIL | N/A

Best trial number: 21
Best F1 score: 0.9019

⚠️ WARNING: Best trial does not have 'fold_scores' attribute!
Make sure you ran training with K-fold cross-validation.


In [4]:
# Check what parameters are actually stored in best trial
best_trial = pipeline.study.best_trial
print("Parameters in best trial:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")
    
print(f"\n🔍 Check if 'numFFLayers' is present: {'numFFLayers' in best_trial.params}")

Parameters in best trial:
  MacroArchitecture: Autoencoder
  window_size: 5
  stride_ratio: 1.0
  aggregation_method: majority_vote
  window_loss_weight: 0.437122852341392
  jitter_strength: 0.0490057537691468
  scaling_range: 0.118295477813213
  time_warp_strength: 0.105871072959977
  globalEmbeddingDim: 18
  globalNumLayers: 3
  globalDropout: 0.310232358024689
  globalHiddenDim_0: 99
  globalHiddenDim_1: 43
  architectureType: Recurrent
  rnnType: LSTM
  hiddenDim: 126
  RecurrentNumLayers: 3
  recurrentDropout: 0.00709611514574818
  numFFLayers: 1
  ffHiddenDim: 249
  ffDropout: 0.315488854421539
  ReconstructionLossWeight: 0.0295996406987581
  RegularizationWeight: 1.05841477613562
  SchedulerPatience: 8
  SchedulerFactor: 0.110966043556621
  SchedulerMinLR: 1.16727119873805e-06
  EarlyStoppingPatience: 16

🔍 Check if 'numFFLayers' is present: True


## Test Checkpoint Search

Check if checkpoints can be found for each fold.

In [5]:
best_trial = pipeline.study.best_trial
n_folds = data_params['n_folds']
studyName = params['project_name'] + params['study_name']

print(f"Searching for checkpoints for trial {best_trial.number} of study {studyName}...\n")

found_checkpoints = []
missing_folds = []

for fold_idx in range(n_folds):
    ckpt_path = pipeline._search_fold_checkpoint(
        trial_number=best_trial.number,
        fold_idx=fold_idx,
        study_name=studyName
    )
    
    if ckpt_path:
        print(f"✓ Fold {fold_idx}: Found checkpoint")
        print(f"  Path: {ckpt_path}")
        found_checkpoints.append(ckpt_path)
    else:
        print(f"✗ Fold {fold_idx}: Checkpoint NOT found (will retrain)")
        missing_folds.append(fold_idx)

print(f"\nSummary:")
print(f"  Found: {len(found_checkpoints)}/{n_folds}")
print(f"  Missing: {len(missing_folds)}/{n_folds}")

if missing_folds:
    print(f"\n⚠️ Folds {missing_folds} will be retrained automatically during ensemble loading.")

Searching for checkpoints for trial 21 of study pirate_pain_classification_theBeastComputerTPE_AugmentedReduced_macro...

✗ Fold 0: Checkpoint NOT found (will retrain)
✗ Fold 1: Checkpoint NOT found (will retrain)
✗ Fold 2: Checkpoint NOT found (will retrain)
✗ Fold 3: Checkpoint NOT found (will retrain)

Summary:
  Found: 0/4
  Missing: 4/4

⚠️ Folds [0, 1, 2, 3] will be retrained automatically during ensemble loading.


## Load Ensemble Model

This will:
1. Search for checkpoints for each fold
2. Load and validate existing checkpoints
3. Retrain any missing or degraded folds
4. Create ensemble with weighted voting (weights = normalized F1 scores)

In [6]:
# print("=" * 70)
# print("LOADING ENSEMBLE WITH FORCE RETRAIN")
# print("=" * 70)
# print("\nThis will skip checkpoint validation and retrain all folds from")
# print("scratch to ensure consistent architecture across all models.\n")

# # Force retrain to ensure consistent architecture across all folds
# ensemble = pipeline.load_ensemble_model(performance_threshold=0.95, force_retrain=False)

# print("\n" + "=" * 70)
# print("[SUCCESS] Ensemble created with consistent architecture!")
# print("=" * 70)

## Generate Submission with Ensemble

This will use the ensemble model to generate predictions for the test set.

In [7]:
print("Generating submission with ensemble model...\n")

# This will automatically use load_ensemble_model() internally
submission_df = pipeline.create_submission()

Generating submission with ensemble model...

Loading K-fold ensemble...

LOADING K-FOLD ENSEMBLE
Best trial: 21
Best mean F1: 0.9019
Architecture: Autoencoder
------------------------------------------------------------
Reconstructing architecture...
✓ Architecture reconstructed

Ensemble details:
  Fold 0: Expected F1 Score = 0.9201
  Fold 1: Expected F1 Score = 0.9463
  Fold 2: Expected F1 Score = 0.8430
  Fold 3: Expected F1 Score = 0.8981
------------------------------------------------------------

[STEP 1/3] Searching for checkpoints...
------------------------------------------------------------
  Fold 0: Not found
  Fold 1: Not found
  Fold 2: Not found
  Fold 3: Not found

Checkpoints found: 0/4

[STEP 2/3] Skipping consistency check (no checkpoints found)

[STEP 3/3] Loading/retraining 4 fold models...
------------------------------------------------------------
Fold 0: Checkpoint not found - retraining...

🔄 Retraining fold 0...
   MacroArchitecture: Autoencoder
   Use wind

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Ti SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


   Train batches: 57, Val batches: 6
   Starting training (max 100 epochs)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


   Training completed. Best F1: 0.9139
✓ Fold 0 retrained successfully: F1=0.9139
Fold 1: Checkpoint not found - retraining...

🔄 Retraining fold 1...
   MacroArchitecture: Autoencoder
   Use windowing: True
   Setting up fold data...
   Train batches: 57, Val batches: 6
   Starting training (max 100 epochs)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


   Training completed. Best F1: 0.9398
✓ Fold 1 retrained successfully: F1=0.9398
Fold 2: Checkpoint not found - retraining...

🔄 Retraining fold 2...
   MacroArchitecture: Autoencoder
   Use windowing: True
   Setting up fold data...
   Train batches: 57, Val batches: 6
   Starting training (max 100 epochs)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


   Training completed. Best F1: 0.7926
✓ Fold 2 retrained successfully: F1=0.7926
Fold 3: Checkpoint not found - retraining...

🔄 Retraining fold 3...
   MacroArchitecture: Autoencoder
   Use windowing: True
   Setting up fold data...
   Train batches: 57, Val batches: 6
   Starting training (max 100 epochs)...
   Training completed. Best F1: 0.8784
✓ Fold 3 retrained successfully: F1=0.8784
------------------------------------------------------------
[OK] All 4 models loaded/retrained
Fold F1 scores: ['0.9139', '0.9398', '0.7926', '0.8784']

✓ Ensemble created with performance-based weights
Weights: ['0.259', '0.267', '0.225', '0.249']
Device: cuda
✓ Submission generated: ../Submissions/submission_20251115_200402.csv
Total predictions: 1324


In [8]:

print(f"\nSubmission preview:")
print(submission_df.head(10))
print(f"\nSubmission shape: {submission_df.shape}")
print(f"Unique predictions: {submission_df['label'].unique()}")
print(f"Class distribution:")
print(submission_df['label'].value_counts())


Submission preview:
  sample_index    label
0          000  no_pain
1          001  no_pain
2          002  no_pain
3          003  no_pain
4          004  no_pain
5          005  no_pain
6          006  no_pain
7          007  no_pain
8          008  no_pain
9          009  no_pain

Submission shape: (1324, 2)
Unique predictions: ['no_pain' 'low_pain' 'high_pain']
Class distribution:
label
no_pain      1001
low_pain      187
high_pain     136
Name: count, dtype: int64


## Test Summary

✅ **What was tested:**
1. Checkpoint search across multiple locations
2. Ensemble model loading with K-fold checkpoints
3. Weight calculation (normalized F1 scores)
4. Ensemble inference (weighted voting)
5. Submission generation with ensemble

⚠️ **If any folds were retrained:**
- Check the logs above for "Retraining fold X" messages
- Retraining is automatic and uses the same hyperparameters
- Retrained models are saved as checkpoints for future use

📊 **Expected behavior:**
- All K models should be loaded (or retrained if missing)
- Weights should sum to 1.0
- Higher performing folds should have higher weights
- Ensemble predictions should be stable and reproducible

## 🔍 Analyze Performance Gap

Let's investigate why validation F1 (~0.92-0.94) is higher than test F1 (0.91)

In [9]:
# 1. Check best trial hyperparameters
print("=" * 70)
print("BEST TRIAL HYPERPARAMETERS")
print("=" * 70)

best_trial = pipeline.study.best_trial
print(f"\nTrial: {best_trial.number}")
print(f"Validation F1: {best_trial.value:.4f}")

# Categorize parameters
print("\n📐 ARCHITECTURE:")
arch_params = ['MacroArchitecture', 'architectureType', 'use_windowing', 'window_size', 'stride_ratio']
for param in arch_params:
    if param in best_trial.params:
        print(f"  {param}: {best_trial.params[param]}")

print("\n🎛️ MODEL CONFIGURATION:")
model_params = ['numLayers', 'hiddenDim', 'isBidirectional', 'numFFLayers', 'ffHiddenDim',
                'globalNumLayers', 'globalEmbeddingDim', 'num_conv_layers', 'initial_channels']
for param in model_params:
    if param in best_trial.params:
        print(f"  {param}: {best_trial.params[param]}")

print("\n🔧 REGULARIZATION:")
reg_params = ['dropout', 'ffDropout', 'globalDropout', 'RegularizationWeight']
for param in reg_params:
    if param in best_trial.params:
        print(f"  {param}: {best_trial.params[param]}")

print("\n📚 TRAINING:")
train_params = ['LearningRate', 'SchedulerType', 'EarlyStoppingPatience', 'ReconstructionLossWeight']
for param in train_params:
    if param in best_trial.params:
        print(f"  {param}: {best_trial.params[param]}")

print("\n" + "=" * 70)

BEST TRIAL HYPERPARAMETERS

Trial: 21
Validation F1: 0.9019

📐 ARCHITECTURE:
  MacroArchitecture: Autoencoder
  architectureType: Recurrent
  window_size: 5
  stride_ratio: 1.0

🎛️ MODEL CONFIGURATION:
  hiddenDim: 126
  numFFLayers: 1
  ffHiddenDim: 249
  globalNumLayers: 3
  globalEmbeddingDim: 18

🔧 REGULARIZATION:
  ffDropout: 0.315488854421539
  globalDropout: 0.310232358024689
  RegularizationWeight: 1.05841477613562

📚 TRAINING:
  EarlyStoppingPatience: 16
  ReconstructionLossWeight: 0.0295996406987581



In [10]:
# 2. Analyze dataset statistics to check for distribution mismatch
print("=" * 70)
print("DATASET STATISTICS COMPARISON")
print("=" * 70)

import pandas as pd
import numpy as np

# Load datasets
train_df = pd.read_csv("../dataset/PirateProcessed/pirate_pain_train.csv")
test_df = pd.read_csv("../dataset/PirateProcessed/pirate_pain_test.csv")
train_labels = pd.read_csv("../dataset/PirateProcessed/pirate_pain_train_labels.csv")

print(f"\n📊 DATASET SIZES:")
print(f"  Training samples: {train_df['sample_index'].nunique()}")
print(f"  Test samples: {test_df['sample_index'].nunique()}")

# Check feature statistics
feature_cols = [col for col in train_df.columns if col not in ['sample_index', 'time']]

print(f"\n📈 FEATURE STATISTICS (comparing train vs test):")
print(f"{'Feature':<20} {'Train Mean':<12} {'Test Mean':<12} {'Difference':<12}")
print("-" * 60)

large_differences = []
for col in feature_cols[:10]:  # Check first 10 features
    train_mean = train_df[col].mean()
    test_mean = test_df[col].mean()
    diff = abs(train_mean - test_mean) / (train_mean + 1e-8)
    
    print(f"{col:<20} {train_mean:<12.4f} {test_mean:<12.4f} {diff:<12.2%}")
    
    if diff > 0.1:  # More than 10% difference
        large_differences.append((col, diff))

if large_differences:
    print(f"\n⚠️ LARGE DIFFERENCES detected in {len(large_differences)} features:")
    for col, diff in large_differences[:5]:
        print(f"  {col}: {diff:.1%} difference")
else:
    print(f"\n✓ No large distribution mismatches detected")

DATASET STATISTICS COMPARISON

📊 DATASET SIZES:
  Training samples: 661
  Test samples: 1324

📈 FEATURE STATISTICS (comparing train vs test):
Feature              Train Mean   Test Mean    Difference  
------------------------------------------------------------
pain_survey_1        0.0000       -0.0269      269069942.77%
pain_survey_2        -0.0000      0.0161       160898656.51%
pain_survey_3        -0.0000      0.0058       57910937.85%
pain_survey_4        -0.0000      0.0002       2410013.39% 
joint_00             0.0000       0.2030       2029765952.81%
joint_01             -0.0000      0.4320       4319538540.65%
joint_02             0.0000       0.1841       1841082199.45%
joint_03             0.0000       0.2396       2396089547.40%
joint_04             0.0000       0.0439       438975777.98%
joint_05             0.0000       0.1430       1429686848.19%

⚠️ LARGE DIFFERENCES detected in 10 features:
  pain_survey_1: 269069942.8% difference
  pain_survey_2: 160898656.5% differ

In [11]:
# 3. Check class distribution
print("=" * 70)
print("CLASS DISTRIBUTION ANALYSIS")
print("=" * 70)

# Training labels distribution
print("\n📊 TRAINING SET CLASS DISTRIBUTION:")
label_counts = train_labels['label'].value_counts().sort_index()
for label, count in label_counts.items():
    percentage = count / len(train_labels) * 100
    print(f"  {label}: {count} samples ({percentage:.1f}%)")

# Submission predictions distribution
submission_files = sorted([f for f in os.listdir('../Submissions/') if f.endswith('.csv')])
if submission_files:
    latest_submission = os.path.join('../Submissions/', submission_files[-1])
    submission_df = pd.read_csv(latest_submission)
    
    print(f"\n📊 TEST PREDICTIONS DISTRIBUTION (latest submission):")
    pred_counts = submission_df['label'].value_counts().sort_index()
    for label, count in pred_counts.items():
        percentage = count / len(submission_df) * 100
        print(f"  {label}: {count} samples ({percentage:.1f}%)")
    
    # Check if distributions are very different
    print("\n🔍 DISTRIBUTION COMPARISON:")
    for label in label_counts.index:
        train_pct = label_counts[label] / len(train_labels) * 100
        test_pct = pred_counts.get(label, 0) / len(submission_df) * 100
        diff = abs(train_pct - test_pct)
        status = "⚠️" if diff > 10 else "✓"
        print(f"  {status} {label}: Train={train_pct:.1f}%, Test={test_pct:.1f}% (diff={diff:.1f}%)")

CLASS DISTRIBUTION ANALYSIS

📊 TRAINING SET CLASS DISTRIBUTION:
  high_pain: 56 samples (8.5%)
  low_pain: 94 samples (14.2%)
  no_pain: 511 samples (77.3%)

📊 TEST PREDICTIONS DISTRIBUTION (latest submission):
  high_pain: 136 samples (10.3%)
  low_pain: 187 samples (14.1%)
  no_pain: 1001 samples (75.6%)

🔍 DISTRIBUTION COMPARISON:
  ✓ high_pain: Train=8.5%, Test=10.3% (diff=1.8%)
  ✓ low_pain: Train=14.2%, Test=14.1% (diff=0.1%)
  ✓ no_pain: Train=77.3%, Test=75.6% (diff=1.7%)


## 💡 Recommended Improvements

Based on the analysis above, here are strategies to close the validation-test gap:

### 🎯 Priority Fixes (Try these first):

1. **Increase Regularization**
   - Increase dropout rates (try 0.3-0.5)
   - Increase weight decay (`RegularizationWeight`)
   - Add batch normalization

2. **Simplify Model** (if architecture is complex)
   - Reduce number of layers
   - Reduce hidden dimensions
   - Try simpler architecture type

3. **Better Data Augmentation**
   - Add noise augmentation for time series
   - Try mixup or cutmix
   - Temporal jittering

4. **Stronger Early Stopping**
   - Increase `EarlyStoppingPatience` to avoid stopping too early
   - Monitor validation F1 instead of loss

### 🔬 Advanced Strategies:

5. **Pseudo-Labeling**
   - Use test predictions with high confidence to augment training
   - Iterative refinement

6. **Better Ensemble Strategy**
   - Try different aggregation methods (soft voting vs hard voting)
   - Add model diversity (train with different random seeds)

7. **Feature Engineering**
   - Add more global features
   - Better feature selection
   - Domain-specific features for pain detection

8. **Cross-Validation Strategy**
   - Try stratified K-fold if not already using
   - Increase number of folds (5-10)
   - Ensure validation mimics test distribution

### 📊 Check the analysis above to identify:
- If specific features have large train/test differences
- If class distributions are imbalanced
- If regularization is too weak
- If architecture is too complex

## 🚨 CRITICAL ISSUE: Preprocessing Mismatch Detected!

**The main problem is NOT the model - it's data preprocessing!**

Your training data has been normalized to ~0 but test data hasn't been normalized the same way. This causes the huge performance gap.

### ✅ Immediate Fix Required:

In [12]:
# Check your preprocessing pipeline
print("=" * 70)
print("INVESTIGATING PREPROCESSING")
print("=" * 70)

# Check if preprocessing was applied consistently
import yaml

# Check if there's a preprocessing params file
preprocessing_params_path = "../Notebook/Params/preprocessing_params.yaml"
if os.path.exists(preprocessing_params_path):
    with open(preprocessing_params_path, 'r') as f:
        preprocess_params = yaml.safe_load(f)
    print("\n✓ Found preprocessing params:")
    print(preprocess_params)
else:
    print("\n⚠️ No preprocessing_params.yaml found!")

# Check actual statistics of loaded data
print("\n📊 ACTUAL FEATURE STATISTICS (first 5 features):")
print(f"{'Feature':<20} {'Train Mean':<15} {'Train Std':<15} {'Test Mean':<15} {'Test Std':<15}")
print("-" * 85)

for col in feature_cols[:5]:
    train_mean = train_df[col].mean()
    train_std = train_df[col].std()
    test_mean = test_df[col].mean()
    test_std = test_df[col].std()
    print(f"{col:<20} {train_mean:<15.6f} {train_std:<15.6f} {test_mean:<15.6f} {test_std:<15.6f}")

print("\n" + "=" * 70)
print("DIAGNOSIS:")
print("=" * 70)
print("\n⚠️ Your training data appears to be normalized (mean≈0)")
print("⚠️ But test data is NOT normalized (mean≠0)")
print("\n💡 SOLUTION:")
print("1. Check preprocessing_params.yaml")
print("2. Make sure BOTH train and test use the SAME normalization")
print("3. Use train statistics to normalize test data")
print("4. Re-run preprocessing on test data with saved train statistics")

INVESTIGATING PREPROCESSING

✓ Found preprocessing params:
{'path_raw_data': '../dataset/Pirate', 'path_processed_data': '../dataset/PirateProcessed', 'name_train_file': 'pirate_pain_train.csv', 'name_test_file': 'pirate_pain_test.csv', 'name_train_labels_file': 'pirate_pain_train_labels.csv', 'drop_all_is_pirate': False, 'one_hot_encode': False, 'PCA': False, 'explained_variance': 0.98, 'feature_selection': False, 'feature_selected': 'None', 'verbose': True, 'columns_excluded_from_normalization': ['sample_index', 'time', 'isPirate', 'isNotPirate']}

📊 ACTUAL FEATURE STATISTICS (first 5 features):
Feature              Train Mean      Train Std       Test Mean       Test Std       
-------------------------------------------------------------------------------------
pain_survey_1        0.000000        1.000000        -0.026907       1.030973       
pain_survey_2        -0.000000       1.000000        0.016090        0.985605       
pain_survey_3        -0.000000       1.000000        0

## 🔧 FIX: Re-preprocess Test Data with Training Statistics

**Root cause:** Test data wasn't normalized using the same statistics as training data.

**Solution:** Re-run preprocessing on test data using the saved normalization parameters from training.

In [13]:
# ACTION REQUIRED: Go back to preprocessing.ipynb and ensure:
# 1. Training data is preprocessed first
# 2. PreProcessor SAVES the normalization statistics (mean/std)
# 3. Test data is preprocessed using the SAVED training statistics

print("=" * 70)
print("QUICK FIX CHECK")
print("=" * 70)

# Check if PreProcessor saved normalization statistics
import glob

preprocessor_files = glob.glob("../dataset/PirateProcessed/*.pkl") + glob.glob("../dataset/PirateProcessed/*.yaml")
print(f"\n📁 Files in PirateProcessed directory:")
for f in preprocessor_files:
    print(f"  {os.path.basename(f)}")

# Check for normalization stats file
norm_stats_patterns = ["*norm*.pkl", "*scaler*.pkl", "*stats*.yaml", "*preprocessing*.yaml"]
found_stats = []
for pattern in norm_stats_patterns:
    found_stats.extend(glob.glob(f"../dataset/PirateProcessed/{pattern}"))

if found_stats:
    print(f"\n✓ Found normalization statistics files:")
    for f in found_stats:
        print(f"  {os.path.basename(f)}")
else:
    print(f"\n⚠️ NO normalization statistics found!")
    print("\n💡 ACTION NEEDED:")
    print("1. Go to Notebook/preprocessing.ipynb")
    print("2. Re-run preprocessing ensuring it saves normalization stats")
    print("3. Make sure test data uses the SAME normalization as training")
    
print("\n" + "=" * 70)

QUICK FIX CHECK

📁 Files in PirateProcessed directory:
  class_weights.yaml

⚠️ NO normalization statistics found!

💡 ACTION NEEDED:
1. Go to Notebook/preprocessing.ipynb
2. Re-run preprocessing ensuring it saves normalization stats
3. Make sure test data uses the SAME normalization as training



## 📋 SUMMARY: What to Do Next

### 🎯 The Problem:
Your **test data is not properly normalized**. The training data has mean≈0 (normalized), but test data has its original scale. This causes the ~6% performance gap.

### ✅ The Solution (3 Options):

#### **Option 1: Re-run Preprocessing (Recommended)**
1. Go to `Notebook/preprocessing.ipynb`
2. Run the preprocessing on BOTH train and test together
3. This ensures test uses training statistics for normalization
4. **Verify** train and test have similar means (~0) after preprocessing

#### **Option 2: Manual Fix (Quick)**

In [14]:
# OPTION 2: Quick manual normalization fix
# This applies training statistics to normalize test data
print("=" * 70)
print("MANUAL NORMALIZATION FIX")
print("=" * 70)

# Load raw data
train_raw = pd.read_csv("../dataset/PirateProcessed/pirate_pain_train.csv")
test_raw = pd.read_csv("../dataset/PirateProcessed/pirate_pain_test.csv")

# Columns to exclude from normalization
exclude_cols = ['sample_index', 'time', 'isPirate', 'isNotPirate']

# Get numeric columns
numeric_cols = [col for col in train_raw.select_dtypes(include=[np.number]).columns 
                if col not in exclude_cols]

print(f"\n📊 Normalizing {len(numeric_cols)} features...")

# Calculate statistics from TRAINING data
train_means = train_raw[numeric_cols].mean()
train_stds = train_raw[numeric_cols].std()

# Apply normalization to TEST data using TRAINING statistics
test_normalized = test_raw.copy()
for col in numeric_cols:
    mean = train_means[col]
    std = train_stds[col]
    
    if std == 0 or np.isnan(std):
        test_normalized[col] = test_raw[col] - mean
    else:
        test_normalized[col] = (test_raw[col] - mean) / std

# Save normalized test data
output_path = "../dataset/PirateProcessed/pirate_pain_test_FIXED.csv"
test_normalized.to_csv(output_path, index=False)

print(f"\n✅ Normalized test data saved to: {output_path}")
print("\n📊 Verification (first 5 features after normalization):")
print(f"{'Feature':<20} {'Test Mean':<15} {'Test Std':<15}")
print("-" * 50)
for col in numeric_cols[:5]:
    print(f"{col:<20} {test_normalized[col].mean():<15.6f} {test_normalized[col].std():<15.6f}")

print("\n⚠️ Now update your pipeline to use 'pirate_pain_test_FIXED.csv' instead!")
print("=" * 70)

MANUAL NORMALIZATION FIX

📊 Normalizing 34 features...

✅ Normalized test data saved to: ../dataset/PirateProcessed/pirate_pain_test_FIXED.csv

📊 Verification (first 5 features after normalization):
Feature              Test Mean       Test Std       
--------------------------------------------------
pain_survey_1        -0.026907       1.030973       
pain_survey_2        0.016090        0.985605       
pain_survey_3        0.005791        0.999083       
pain_survey_4        0.000241        1.001032       
joint_00             0.202977        1.013327       

⚠️ Now update your pipeline to use 'pirate_pain_test_FIXED.csv' instead!


### 🎯 After Fixing Preprocessing:

**Expected improvements:**
- Test F1 should increase from **0.91 → ~0.94-0.95**
- Closing the 6% gap caused by data mismatch
- Your model is actually good! It just needs properly normalized test data

**Additional optimizations to try AFTER fixing preprocessing:**
1. **Increase dropout** in feedforward (currently 0.16 → try 0.3-0.4)
2. **Test-Time Augmentation (TTA)** - add small noise to test data, average predictions
3. **Better ensemble weighting** - try temperature scaling
4. **More folds** - increase from 4 to 5-10 folds for better stability

**Why your current model is actually good:**
- ✅ High validation F1 (0.957)
- ✅ Good regularization (weight decay 10.0, global dropout 0.5)
- ✅ Reasonable architecture (3-layer LSTM with windowing)
- ✅ Class distributions are balanced
- ❌ **ONLY problem:** Test data wasn't normalized properly!